# 1. Transformer와 GPT의 차이

### 1. Encoder 블록 제거
- 트랜스포머: Encoder + Decoder
- GPT: Decoder
언어 모델은 앞 토큰을 가지고 다음 토큰을 예측만 해도 되기 때문에 인코더 없이도 동작하는 것을 확인했다. 따라서 GPT는 Decoder 블록만 여러 층 쌓은 구조로 만들어졌다.

### 2. Encoder-Decoder Cross Attention 제거
트랜스포머의 Decoder에 있던 Cross Attention은 Encoder의 출력도 참고하기 때문에, Encoder를 제거하면서 이 어텐션도 함께 제거되었다.

### 3. Self-Attention은 항상 Masked Attention으로 사용
트랜스포머의 Encoder는 양방향(Bidirectional)으로 모든 토큰을 참조하지만,  GPT는 Decoder 구조를 따르므로 항상 미래 정보를 차단하는 Masked Self-Attention만 사용한다.

### 4. 입력 처리 방식 단순화
트랜스포머에서는 source sentence와 target sentence 두가지 입력이 존재하지만,  
GPT에서는 여러 sentence들의 중간에 Delimeter를 사용해서 하나의 연속된 시퀀스로 만든다.  
따라서 추론, 질문 응답, 텍스트 함의 등 다양한 분야의 입력들이 모두 단일 텍스트 시퀀스 문제로 변환된다.

### 5. 출력 구조 변경
트랜스포머는 출력층이 하나뿐이지만, 
GPT(특히 GPT-1/2)는 목적에 따라 출력층을 갈아 끼우기 쉬운 구조로 설계되었다.
트랜스포머는 번역용 출력 헤드 하나에 고정되어 있다.
GPT는 Language Modeling Head, Classification Head 등이 있다.

### 6. 파라미터 공유
GPT에서는 입력 임베딩 층과 출력층의 가중치 행렬을 똑같이 공유하는 경우가 많다. -> weight tying  
트랜스포머 논문에서는 선택 사항이었지만 GPT계열에서는 거의 표준이다.

### 7. 위치 임베딩 방식 차이
- 트랜스포머: sinusoidal positional encoding
    - 고정식. 사인 코사인 함수를 사용하여 위치 값 계산
- GPT: learned positional embedding
    - 학습식. 위치 정보 자체도 하나의 학습 가능한 파라미터로 취급함. 모델이 학습과정에서 위치 정보를 스스로 최적화 함.
    - 장점: 특정 도메인이나 문장 구조에 가장 적합한 위치 표현을 모델이 직접 찾아내므로, 성능상 유리할 수 있음.
    - 단점: 학습시 설정한 max seq len을 넘어서는 위치에 대해서는 대응하기가 어려움. (예. 512 토큰까지 학습했다면, 513번째 토큰의 위치 정보는 알 수 없음)

# 2. GPT 구현

In [1]:
!pip install sentencepiece nltk

In [2]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.7.1+cu118


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2.1 데이터 준비

In [4]:
# 경로 설정
BASE_DIR = "./"
MODEL_DIR = os.path.join(BASE_DIR, "models")
TOKENIZER_DIR = os.path.join(BASE_DIR, "tokenizers")
DATA_DIR = os.path.join(BASE_DIR, "data")

for path in [MODEL_DIR, TOKENIZER_DIR, DATA_DIR]:
    os.makedirs(path, exist_ok=True)

In [5]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

zip_filename = os.path.join(DATA_DIR, zip_filename)
urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

### 전처리
중복 제거

In [6]:
extracted_folder = os.path.join(DATA_DIR, "spa-eng")
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r") as f:
    spa_eng_sentences = f.read().splitlines()

spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)
print("Example:", total_sentence_count)

for sen in spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Example: 118964
>> Today our artificial satellites are revolving around the earth.	Hoy, nuestros satélites artificiales están girando alrededor de la Tierra.
>> Tom looks a bit tired.	Tom se ve un poco cansado.
>> The population of this country is smaller than that of the United States.	La población de este país es más pequeña que la de Estados Unidos.
>> She looked at the picture to refresh her memory.	Ella miró la fotografía para refrescar su memoria.
>> The store closes at eleven.	La tienda cierra a las once.


전처리 함수 정의 후 전처리

In [7]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

### 데이터 분할
영어와 스페인어 분리

In [8]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]
print("Train Example:", len(train_spa_eng_sentences))
for sen in train_spa_eng_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_spa_eng_sentences))
for sen in test_spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  594


Train Example: 118370
>> today our artificial satellites are revolving around the earth.	hoy, nuestros satélites artificiales están girando alrededor de la tierra.
>> tom looks a bit tired.	tom se ve un poco cansado.
>> the population of this country is smaller than that of the united states.	la población de este país es más pequeña que la de estados unidos.
>> she looked at the picture to refresh her memory.	ella miró la fotografía para refrescar su memoria.
>> the store closes at eleven.	la tienda cierra a las once.


Test Example: 594
>> i want you to help me find out who stole my car.	quiero que me ayudes a descubrir quién me robó el coche.
>> they signed the peace treaty.	ellos firmaron el tratado de paz.
>> they do many things together.	ellos hacen muchas cosas juntos.
>> i can't connect to the internet.	no puedo conectarme a internet.
>> tom is playing with his yo-yo.	tom está jugando con su yo-yo.


학습 데이터와 테스트 데이터 나눠주기 -> 비지도 학습인데 필요가 있나?

In [9]:
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences = []
    eng_sentences = []
    for spa_eng_sentence in tqdm(spa_eng_sentences):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
print(len(train_eng_sentences))
print(train_eng_sentences[0])
print('\n')
print(len(train_spa_sentences))
print(train_spa_sentences[0])

test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)
print(len(test_eng_sentences))
print(test_eng_sentences[0])
print('\n')
print(len(test_spa_sentences))
print(test_spa_sentences[0])

  0%|          | 0/118370 [00:00<?, ?it/s]

118370
today our artificial satellites are revolving around the earth.


118370
hoy, nuestros satélites artificiales están girando alrededor de la tierra.


  0%|          | 0/594 [00:00<?, ?it/s]

594
i want you to help me find out who stole my car.


594
quiero que me ayudes a descubrir quién me robó el coche.


### 토큰화

In [10]:
# Sentencepiece 기반의 토크나이저
def generate_tokenizer(corpus,
                       vocab_size,
                       lang,
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = f"{DATA_DIR}/{lang}_corpus.txt"
    model = f"{TOKENIZER_DIR}/{lang}_spm"
    print(model)
    
    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    # spm.SentencePieceTrainer.Train(
    #     '--input=./%s --model_prefix=%s --vocab_size=%d'\
    #     % (file, model, vocab_size) + \
    #     '--pad_id==%d --bos_id=%d --eos_id=%d --unk_id=%d'\
    #     % (pad_id, bos_id, eos_id, unk_id)
    # )

    tokenizer_prefix = os.path.join(TOKENIZER_DIR, "eng_only")

    spm.SentencePieceTrainer.Train(
        input=file,
        model_prefix=model,
        vocab_size=VOCAB_SIZE,
        pad_id=0,
        bos_id=1,  # 2에서 1로 수정
        eos_id=2,  # 3에서 2로 수정
        unk_id=3   # 1에서 3으로 수정
    )
    
    tokenizer = spm.SentencePieceProcessor()
    print(f"{model}.model")
    tokenizer.Load(f"{model}.model")

    return tokenizer

단어 사전 생성

In [11]:
VOCAB_SIZE = 10000  # 20000에서 수정
# tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng') # 수정 전
tokenizer = generate_tokenizer(train_eng_sentences, VOCAB_SIZE, lang='eng-only') # 수정 후

# 트랜스포머용이므로 제거
# 토크나이저가 encode_as_ids()를 부를 때마다 무조건 자동으로 BOS/EOS를 붙임
# 학습용, 추론용 프롬프트 인코딩 모두 똑같이 적용되버림
# tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

./tokenizers/eng-only_spm
./tokenizers/eng-only_spm.model


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./data/eng-only_corpus.txt
  input_format: 
  model_prefix: ./tokenizers/eng-only_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_

토크나이저로 토큰화 하기

In [12]:
"""
# 수정 전
def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)  
        corpus.append(tokens)
    return corpus
"""

def make_corpus(sentences, tokenizer):
    corpus = []
    bos_id = tokenizer.bos_id()
    eos_id = tokenizer.eos_id()

    for sentence in sentences:
        token_ids = tokenizer.encode_as_ids(sentence)
        token_ids = [bos_id] + token_ids + [eos_id]
        corpus.append(token_ids)
    return corpus

# 영어
eng_corpus = make_corpus(train_eng_sentences, tokenizer)

# 스페인어
# spa_corpus = make_corpus(train_spa_sentences, tokenizer)

In [13]:
print(make_corpus(["there are many people"], tokenizer)[0])

[1, 53, 34, 128, 122, 2]


In [14]:
# def make_corpus(sentences, tokenizer, add_bos=True, add_eos=True):
#     corpus = []
#     bos_id = tokenizer.bos_id()
#     eos_id = tokenizer.eos_id()

#     for sentence in tqdm(sentences):
#         tokens = tokenizer.encode_as_ids(sentence)

#         if add_bos and bos_id != -1:
#             tokens = [bos_id] + tokens
#         if add_eos and eos_id != -1:
#             tokens = tokens + [eos_id]

#         corpus.append(tokens)

#     return corpus

# eng_corpus = make_corpus(train_eng_sentences, tokenizer, add_bos=True, add_eos=True)

In [15]:
# 토큰화 확인
print(train_eng_sentences[0])
print(eng_corpus[0])

# print(train_spa_sentences[0])
# print(spa_corpus[0])

today our artificial satellites are revolving around the earth.
[1, 139, 124, 4687, 4067, 15, 34, 9697, 82, 303, 7, 1004, 4, 2]


### 데이터셋 만들기
한 문장의 토큰 길이는 50으로

In [16]:
# 한 문장의 토큰 길이
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

# enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)  # 수정 전
dec_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)  # 수정 후

# dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)  # 제거

# print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]

torch.Size([118370, 50])


훈련용 데이터 셋 -> 배치 크기의 텐서로 만들기

In [17]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TensorDataset(enc_ndarray, dec_ndarray)를 사용했기 때문에
# 각 샘플은 (src, tgt) 형태의 튜플로 반환된다.
# DataLoader에서 배치를 만들면 (src_batch, tgt_batch)가 된다.
# train_dataset = TensorDataset(enc_ndarray, dec_ndarray)  # 수정 전
train_dataset = TensorDataset(dec_ndarray)  # 수정 후 -> tgt : [batch_size, seq_len]

# 여기서 train_dataset이 기존 트래스포머 구조에서는 튜플 형태로 들어감 (src, tgt)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

## 2.2 모델 구현

위치 인코딩  
GPT는 트랜스포머와 달리 sinusoidal이 아닌 learned positional embedding을 사용한다.

In [18]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

마스크 생성

In [19]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


# def generate_masks(src: torch.Tensor, tgt: torch.Tensor):  # 수정 전
def generate_masks(tgt: torch.Tensor):  # 수정 후
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    # enc_mask = generate_padding_mask(src)  # 제거
    
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    # dec_enc_mask = generate_padding_mask(src)  # 제거

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    # return enc_mask, dec_enc_mask, dec_mask  # 수정 전
    return dec_mask  # 수정 후

멀티 헤드 어텐션

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

Position-wise Feed Forward Network

In [21]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        # self.relu = nn.ReLU()  # 수정 전
        self.gelu = nn.GELU()

    def forward(self, x):
        # out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        # ReLU 대신 GELU를 통과시킵니다.
        out = self.gelu(self.fc1(x))  # 첫 번째 Dense + GEL
        out = self.fc2(out)          # 두 번째 Dense
        return out

---
인코더 레이어 생략  

---
디코더 레이어  
- DecoderLayer 내부 cross-attention 모듈 삭제

In [22]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    # def forward(self, x, enc_out, dec_enc_mask, padding_mask):
    def forward(self, x, padding_mask):  # enc_out, dec_enc_mask 제거
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        """ 제거
        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual
        """

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        # return out, dec_attn, dec_enc_attn 
        return out, dec_attn

In [23]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    # def forward(self, x, enc_out, dec_enc_mask, padding_mask):
    def forward(self, x, padding_mask):  # enc_out, dec_enc_mask 제거
        out = x
        dec_attns = []
        # dec_enc_attns = []  # 제거
        for i in range(self.n_layers):
            # out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask) # 수정 전
            out, dec_attn = self.dec_layers[i](out, padding_mask)  # 수정 후
            dec_attns.append(dec_attn)
            # dec_enc_attns.append(dec_enc_attn)  # 제거
        # return out, dec_attns, dec_enc_attns  # 수정 전
        return out, dec_attns  # 수정 후

### 전체 모델 조립

In [24]:
import math

class Gpt(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=False, shared_emb=False):  # 디코더 only이기 때문에 shared_emb은 사실 필요없다.
        super(Gpt, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        # self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))  # 수정 전
        self.pos_emb = nn.Embedding(pos_len, d_model)  # 수정 후
        
        self.do = nn.Dropout(dropout)

        # self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # 추가
        
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        # out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)  # 수정 전
        out = out + self.pos_emb(positions)  # 수정 후
        out = self.do(out)
        return out

    # def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
    def forward(self, dec_in, dec_mask):  # 제거
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        # enc_in_emb = self.embedding(self.enc_emb, enc_in)  # 통으로 제거
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        # enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)  # 통으로 제거
        # dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask) # 수정 전
        dec_out, dec_attns = self.decoder(dec_in_emb, dec_mask) # 수정 후
        

        logits = self.fc(dec_out)
        # return logits, enc_attns, dec_attns, dec_enc_attns  # 수정 전
        return logits, dec_attns  # 수정 후

### 모델 인스턴스 생성
- 하이퍼 파라미터 정의
- 모델 GPU로 보내기 & 벡터 차원 512로 설정

In [25]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
d_model = 512

gpt = Gpt(
    n_layers=2,
    d_model=d_model,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=False,
    shared_emb=False)

gpt = gpt.to(device)

Learning Rate 스케줄러

In [26]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=4000): # 4000 # 60
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

옵티마이저

In [27]:
# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(gpt.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

손실함수 정의

In [28]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

Train Step 정의
- src: encoder에 들어가는 입력 문장 -> 제거
- tgt_in: decoder에 들어가는 정답 시퀀스의 shifted input

In [29]:
# def train_step(src, tgt, model, optimizer):  # 수정 전
def train_step(tgt, model, optimizer):  # 수정 후
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력  ! 튜플
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    # enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)  # 수정 전
    dec_mask = generate_masks(tgt_in)  # 수정 후

    
    # src = src.to(device)
    tgt_in = tgt_in.to(device)
    # enc_mask = enc_mask.to(device)  # 제거
    # dec_enc_mask = dec_enc_mask.to(device)  # 제거
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    # predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask) # 수정 전
    predictions, dec_attns = model(tgt_in, dec_mask) # 수정 후

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    # return loss, enc_attns, dec_attns, dec_enc_attns  # 수정 전
    return loss, dec_attns  # 수정 후

### 훈련

In [40]:
%%time

EPOCHS = 5

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    # 번역일 때 - 지도 학습일 때
    # for batch, (src, tgt) in enumerate(train_dataloader):   # 수정 전
    # 언어 생성일 때 - 비지도 학습일 때
    for batch, (tgt,) in enumerate(train_dataloader):   # 수정 후  
        # --- 디버깅 코드 시작 (첫 에폭, 첫 배치에서만 실행) ---
        if epoch == 0 and batch == 0:
            print(f"\n" + "="*60)
            print(f"[DEBUG] 현재 설정된 tokenizer.eos_id(): {tokenizer.eos_id()}")
            
            # tgt는 [batch_size, seq_len] 형태이므로 첫 번째 샘플을 가져옵니다.
            sample_tgt = tgt[0].tolist()
            print(f"[DEBUG] 학습 데이터(tgt) 샘플 ID: {sample_tgt}")
            
            # 문장 내에 0(Pad)이 있을 수 있으므로, 패딩을 제외한 실제 내용의 마지막 토큰을 확인합니다.
            # 패딩이 뒤에 붙는 구조라면, 0이 아닌 마지막 숫자를 찾아야 합니다.
            real_tokens = [t for t in sample_tgt if t != 0] # 패딩(0) 제외
            if len(real_tokens) > 0:
                last_token = real_tokens[-1]
                print(f"[DEBUG] 패딩 제외 마지막 토큰 ID: {last_token}")
                
                if last_token == tokenizer.eos_id():
                    print(f"✅ 결과: 정답 데이터 끝에 EOS({tokenizer.eos_id()})가 정상적으로 포함되어 있습니다.")
                else:
                    print(f"❌ 결과: 정답 데이터 끝에 EOS가 없습니다! 모델이 '종료'를 배울 수 없는 상태입니다.")
            print("="*60 + "\n")
        # --- 디버깅 코드 끝 ---

        
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        # loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)  # 수정 전
        loss, dec_attns = train_step(tgt, gpt, optimizer)  # 수정 후

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")

  0%|          | 0/1850 [00:00<?, ?it/s]


[DEBUG] 현재 설정된 tokenizer.eos_id(): 2
[DEBUG] 학습 데이터(tgt) 샘플 ID: [1, 748, 350, 8, 2465, 67, 124, 417, 629, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[DEBUG] 패딩 제외 마지막 토큰 ID: 2
✅ 결과: 정답 데이터 끝에 EOS(2)가 정상적으로 포함되어 있습니다.

Epoch 1, Loss: 6.4625


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 2, Loss: 6.1526


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 3, Loss: 5.9223


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 4, Loss: 5.7495


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 5, Loss: 5.6131
CPU times: user 12min 32s, sys: 4.45 s, total: 12min 37s
Wall time: 10min 20s


모델 저장

In [31]:
# 현재 시간 문자열 생성
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

checkpoint = {
    "model_state": gpt.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch
}

torch.save(checkpoint, os.path.join(MODEL_DIR, f"checkpoint_{timestamp}.pt"))

## 모델 평가

모델 불러오기

In [32]:
# # 1. checkpoint load
# MODEL_FILE = "checkpoint_20260306_0645.pt"

# checkpoint = torch.load(MODEL_FILE, map_location=device)
# gpt.load_state_dict(checkpoint["model_state"])
# gpt.to(device).eval()

평가용 loss / perplexity 계산 코드

In [33]:

@torch.no_grad()
def evaluate_loss_perplexity(model, dataloader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for (tgt,) in dataloader:
        tgt = tgt.to(device)

        tgt_in = tgt[:, :-1]
        gold = tgt[:, 1:]

        dec_mask = generate_masks(tgt_in).to(device)

        outputs = model(tgt_in, dec_mask)

        # 네 기존 코드: return logits, dec_attns
        # 혹은 수정 후 코드: return logits
        if isinstance(outputs, tuple):
            predictions = outputs[0]
        else:
            predictions = outputs

        # token별 loss
        loss_per_token = F.cross_entropy(
            predictions.contiguous().view(-1, predictions.size(-1)),
            gold.contiguous().view(-1),
            reduction='none'
        ).view(gold.size())

        mask = (gold != 0).float()
        loss_sum = (loss_per_token * mask).sum()
        token_count = mask.sum()

        total_loss += loss_sum.item()
        total_tokens += token_count.item()

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)

    return avg_loss, ppl

프롬프트 문장 -> 토큰화

In [34]:
def encode_prompt(prompt, tokenizer):
    bos_id = tokenizer.bos_id()

    # eos는 넣지 않음
    token_ids = tokenizer.encode_as_ids(prompt)

    if bos_id != -1:
        token_ids = [bos_id] + token_ids

    return token_ids

토큰 id를 다시 문장으로 바꾸는 함수

In [35]:
def decode_tokens(token_ids, tokenizer):
    pad_id = tokenizer.pad_id()
    bos_id = tokenizer.bos_id()
    eos_id = tokenizer.eos_id()

    clean_ids = []
    for tid in token_ids:
        if tid in [pad_id, bos_id]:
            continue
        if tid == eos_id:
            break
        clean_ids.append(tid)

    return tokenizer.decode_ids(clean_ids)

### 문장 생성기
다음 토큰을 하나씩 autoregressive하게 생성함

In [36]:
# greedy=True로 하면 매 순간 가장 확률이 높은 단어만 선택하므로 마지막 단어의 무한굴레에 빠질 수 있음
@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=30,
    temperature=1.0,
    top_k=None,
    greedy=False
):
    model.eval()

    eos_id = tokenizer.eos_id()

    input_ids = encode_prompt(prompt, tokenizer)
    generated = torch.tensor([input_ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        dec_mask = generate_masks(generated).to(device)

        outputs = model(generated, dec_mask)

        # 기존 코드: return logits, dec_attns
        # 수정 후 코드: return logits
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        next_token_logits = logits[:, -1, :]

        # temperature 적용
        next_token_logits = next_token_logits / temperature

        if greedy:
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        else:
            if top_k is not None:
                values, indices = torch.topk(next_token_logits, top_k)
                probs = torch.softmax(values, dim=-1)
                sampled_idx = torch.multinomial(probs, num_samples=1)
                next_token = indices.gather(-1, sampled_idx)
            else:
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_token], dim=1)
        
        if next_token.item() == eos_id:
            break

        # 너무 길어지면 방지
        if generated.size(1) >= MAX_LEN:
            break

    output_ids = generated[0].tolist()
    return decode_tokens(output_ids, tokenizer), output_ids

### 테스트

In [42]:
print("=" * 20, "greedy를 False로 했을 때", "=" * 20)

test_prompts = [
    "there are",
    "i am",
    "he is",
    "we can",
    "this is"
]

generated_text, generated_ids = generate_text(
    model=gpt,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=20,
    temperature=1.0,
    greedy=False
)

print("=" * 60)
print("prompt        :", prompt)
print("generated ids :", generated_ids)
print("generated text:", generated_text)

prompt        : there are
generated ids : [1, 53, 34, 8, 8, 4831, 4, 2]
generated text: there are to to confidential.
prompt        : i am
generated ids : [1, 5, 118, 9, 9, 8, 7, 6, 2]
generated text: i am you you to the'
prompt        : he is
generated ids : [1, 16, 14, 11, 5, 5553, 6236, 4, 2]
generated text: he is a i viewsusy.
prompt        : we can
generated ids : [1, 29, 36, 11, 1027, 6643, 7, 7, 858, 4, 2]
generated text: we can a check obstacle the the promised.
prompt        : this is
generated ids : [1, 26, 14, 6, 20, 7, 5812, 9, 9543, 98, 9, 4, 2]
generated text: this is' that the euros you fasci been you.


# 결과 모음

#### 5에포크
1850/1850 [02:03<00:00, 15.86it/s, Batch Loss=6.6681]
Epoch 5, Loss: 6.8422  

디버깅
```text
============================================================
[DEBUG] 현재 설정된 tokenizer.eos_id(): 2
[DEBUG] 학습 데이터(tgt) 샘플 ID: [1, 75, 23, 9, 45, 24, 8, 645, 215, 1961, 15, 8, 124, 3006, 13, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[DEBUG] 패딩 제외 마지막 토큰 ID: 2
✅ 결과: 정답 데이터 끝에 EOS(2)가 정상적으로 포함되어 있습니다.
============================================================
```

결과

```
============================================================
prompt        : there are
generated ids : [1, 53, 34, 4, 2]
generated text: there are.
============================================================
prompt        : i am
generated ids : [1, 5, 118, 1906, 1419, 2]
generated text: i amous math
============================================================
prompt        : he is
generated ids : [1, 16, 14, 9388, 3610, 1136, 5164, 6322, 4, 2]
generated text: he is employer trade apartment kyushuwife.
============================================================
prompt        : we can
generated ids : [1, 29, 36, 2]
generated text: we can
============================================================
prompt        : this is
generated ids : [1, 26, 14, 6, 9320, 476, 4, 2]
generated text: this is' praisi.
```

---
#### 10에포크
디버깅
```text
============================================================
[DEBUG] 현재 설정된 tokenizer.eos_id(): 2
[DEBUG] 학습 데이터(tgt) 샘플 ID: [1, 748, 350, 8, 2465, 67, 124, 417, 629, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[DEBUG] 패딩 제외 마지막 토큰 ID: 2
✅ 결과: 정답 데이터 끝에 EOS(2)가 정상적으로 포함되어 있습니다.
============================================================
```

결과

```
==================== greedy를 False로 했을 때 ====================
prompt: there are
generated ids: [1, 53, 34, 37, 4, 2]
generated text: there are mary.
============================================================
prompt        : there are
generated ids : [1, 53, 34, 8, 8, 4831, 4, 2]
generated text: there are to to confidential.
============================================================
prompt        : i am
generated ids : [1, 5, 118, 9, 9, 8, 7, 6, 2]
generated text: i am you you to the'
============================================================
prompt        : he is
generated ids : [1, 16, 14, 11, 5, 5553, 6236, 4, 2]
generated text: he is a i viewsusy.
============================================================
prompt        : we can
generated ids : [1, 29, 36, 11, 1027, 6643, 7, 7, 858, 4, 2]
generated text: we can a check obstacle the the promised.
============================================================
prompt        : this is
generated ids : [1, 26, 14, 6, 20, 7, 5812, 9, 9543, 98, 9, 4, 2]
generated text: this is' that the euros you fasci been you.
```

# 문제 발생

### 1. 생성 오류 -> 입력과 출력이 동일함

테스트 결과가 다음과 같이 나옴.
```
============================================================
prompt   : there are
generated: there are
============================================================
prompt   : i am
generated: i am
============================================================
prompt   : he is
generated: he is
============================================================
prompt   : we can
generated: we can
============================================================
prompt   : this is
generated: this is
```

원인은 눈에 보이는 새 토큰이 안 붙는 상황.
실제론 한 토큰 생성했는데,
그게 EOS라서 사람이 보기엔 “아무것도 안 생성된 것처럼” 보이는 거.

입력:
[BOS, there, are]

생성:
[BOS, there, are, EOS]

decode할 때 BOS/EOS를 제거하니 최종 문자열이 그냥 원문이 되는 것.

확인을 위해 generated_ids를 같이 출력해봤다.
```text
============================================================
prompt        : there are
generated ids : [1, 1, 52, 33, 2, 2]
generated text: there are
============================================================
prompt        : i am
generated ids : [1, 1, 4, 116, 2, 2]
generated text: i am
============================================================
prompt        : he is
generated ids : [1, 1, 15, 13, 2, 2]
generated text: he is
============================================================
prompt        : we can
generated ids : [1, 1, 29, 35, 2, 2]
generated text: we can
============================================================
prompt        : this is
generated ids : [1, 1, 25, 13, 2, 2]
generated text: this is
```

[BOS, BOS, there, are, EOS, EOS] 이런식으로 출력되고 있었다.
여기서 마지막 `EOS`는 생성된 토큰으로 보인다. 즉 실제 흐름은  
[1, 1, 52, 33, 2]

#### 해결한 방법
tokenizer 단계에서 토큰을 붙이는 부분을 삭제했다. 해당 코드가 전역설정으로, 프롬프트를 인코딩할 때도 불러들여져 그런 것으로 보인다.




In [39]:
print("bos_id:", tokenizer.bos_id())
print("eos_id:", tokenizer.eos_id())
print("pad_id:", tokenizer.pad_id())

bos_id: 1
eos_id: 2
pad_id: 0


### 2. 손실율이 4000 이상이 나옴

```
100%
 1850/1850 [02:00<00:00, 15.46it/s, Batch Loss=11694.8965]
Epoch 1, Loss: 11630.6996
100%
 1850/1850 [01:58<00:00, 16.39it/s, Batch Loss=11578.9043]
Epoch 2, Loss: 11600.5641
```
shared_fc 끔. -> 해결됨

### 3. 단어가 끝없이 나옴

#### 3.1 EOS 토큰 이슈로 모델이 종료를 배우지 못했음.

트랜스포머 구조에서는 토큰 붙이는 코드를 전역으로 처리해두었었다.  
이 코드를 그대로 사용하면서 문제가 발생.  
```python
tokenizer.set_encode_extra_options("bos:eos")
```
---
결과

---
```
============================================================
prompt        : there are
generated ids : [2, 53, 34, 4461, 7271, 1155, 5499, 1186, 7069, 1279, 6039, 313, 3279, 4178, 9750, 6599, 7965, 3931, 6653, 1326, 9273, 9866, 9551]
generated text: there are responded pile project pri asking diar headache toss sleep windyay teller finnish fiftcan hoarse keepspus initiascription
============================================================
prompt        : i am
generated ids : [2, 5, 117, 3859, 8068, 7751, 9948, 6687, 4, 3389, 9985, 3347, 3296, 2333, 5787, 8146, 1082, 5238, 3635, 9927, 1817, 1667, 7425]
generated text: i am extent subtitles tangerinessub specialization. performance sam mt sake earn pharmacy disl soup sunrise thunderonda bald passengers emp
============================================================
prompt        : he is
generated ids : [2, 16, 14, 9894, 5387, 5005, 5742, 4663, 9050, 6258, 4142, 7290, 2258, 2923, 3399, 5433, 2178, 7800, 5560, 2841, 537, 2945, 4849]
generated text: he iserity prosperright celsius frozeexpensive site basicfriend prisoner error victory robots lately dictatesssured consist age luxury foul
============================================================
prompt        : we can
generated ids : [2, 29, 36, 7397, 9880, 5565, 2173, 3762, 6066, 3569, 8005, 9267, 590, 688, 3901, 3806, 3119, 4334, 7112, 111, 7660, 4556, 3480]
generated text: we can mactub shore murdered elbow stink scotland sadnesstexas wine exactly youngest baboon reliable democracyhim us bewildered smith wasted
============================================================
prompt        : this is
generated ids : [2, 26, 14, 3231, 4835, 5794, 4, 5802, 2628, 9918, 4407, 2248, 259, 6460, 9474, 3746, 3164, 2370, 9633, 6510, 1490, 5991, 8943]
generated text: this is hurried protection shinano. yogurt lamp hypoc reflection towercause fre extraordinar produce asks sweating aspir elaborate diet graceful imper
```

#### 3.2 greedy 옵션이 True일 때 단어를 max_len까지 출력

```text
==================== greedy를 False로 했을 때 ====================
prompt: there are
generated ids: [1, 53, 34, 3740, 8575, 4369, 1535, 2575, 5722, 4553, 4011, 7389, 5168, 1842, 2355, 2242, 745, 4769, 3249, 632, 2958, 9567, 4444]
generated text: there are pace washe pickpocket distance tourists acquitted formed opponentfelt noodles army bootsried deal groceries embarrassing expensive racket valen childish

 ==================== greedy를 True로 했을 때 ====================
prompt: there are
generated ids: [1, 53, 34, 4, 2]
generated text: there are.
```

EOS 토큰의 문제
학습 중간에 디버깅 코드를 삽입해서 EOS를 확인해봄.

---
```python
%%time

EPOCHS = 5

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    # 번역일 때 - 지도 학습일 때
    # for batch, (src, tgt) in enumerate(train_dataloader):   # 수정 전
    # 언어 생성일 때 - 비지도 학습일 때
    for batch, (tgt,) in enumerate(train_dataloader):   # 수정 후  
        # --- 디버깅 코드 시작 (첫 에폭, 첫 배치에서만 실행) ---
        if epoch == 0 and batch == 0:
            print(f"\n" + "="*60)
            print(f"[DEBUG] 현재 설정된 tokenizer.eos_id(): {tokenizer.eos_id()}")
            
            # tgt는 [batch_size, seq_len] 형태이므로 첫 번째 샘플을 가져옵니다.
            sample_tgt = tgt[0].tolist()
            print(f"[DEBUG] 학습 데이터(tgt) 샘플 ID: {sample_tgt}")
            
            # 문장 내에 0(Pad)이 있을 수 있으므로, 패딩을 제외한 실제 내용의 마지막 토큰을 확인합니다.
            # 패딩이 뒤에 붙는 구조라면, 0이 아닌 마지막 숫자를 찾아야 합니다.
            real_tokens = [t for t in sample_tgt if t != 0] # 패딩(0) 제외
            if len(real_tokens) > 0:
                last_token = real_tokens[-1]
                print(f"[DEBUG] 패딩 제외 마지막 토큰 ID: {last_token}")
                
                if last_token == tokenizer.eos_id():
                    print(f"✅ 결과: 정답 데이터 끝에 EOS({tokenizer.eos_id()})가 정상적으로 포함되어 있습니다.")
                else:
                    print(f"❌ 결과: 정답 데이터 끝에 EOS가 없습니다! 모델이 '종료'를 배울 수 없는 상태입니다.")
            print("="*60 + "\n")
        # --- 디버깅 코드 끝 ---

        
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        # loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)  # 수정 전
        loss, dec_attns = train_step(tgt, gpt, optimizer)  # 수정 후

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")
```
---
결과

---
```text

============================================================
[DEBUG] 현재 설정된 tokenizer.eos_id(): 2
[DEBUG] 학습 데이터(tgt) 샘플 ID: [10, 170, 6, 12, 52, 255, 77, 16, 6, 15, 813, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[DEBUG] 패딩 제외 마지막 토큰 ID: 4
❌ 결과: 정답 데이터 끝에 EOS가 없습니다! 모델이 '종료'를 배울 수 없는 상태입니다.
============================================================
```


1. 결과 분석  
    현재 설정된 EOS ID: 2  

    실제 데이터의 끝: 4 (마침표나 일반 단어로 추정됨)  

    상태: 모델은 학습 데이터 전체를 통틀어 **"2번 토큰이 나오면 문장이 끝난다"**는 사실을 단 한 번도 본 적이 없습니다. 그래서 생성할 때도 2번(EOS)을 내뱉을 확률이 0에 가깝고, 결국 멈추지 못하는 것입니다.  

2. 해결 방법: 데이터 전처리 코드 수정  
    PDF 내의 데이터 준비 로직(Dataset을 정의하거나 데이터를 토큰화하는 부분)을 찾아보세요. 보통 문장을 토큰 ID로 변환한 직후에 EOS를 수동으로 추가해줘야 합니다.